In [5]:

import numpy as np

# ---------- INPUT ----------
num_staff = int(input("Enter number of staff: "))
staff_names = [
    input(f"Enter name of staff {i + 1}: ")
    for i in range(num_staff)
]

num_days = int(input("Enter number of days: "))
num_shifts = int(input("Enter number of shifts per day: "))

shift_names = [
    input(f"Enter name of shift {i + 1}: ")
    for i in range(num_shifts)
]

# Required staff for each shift
required = []
for shift in shift_names:
    required.append(
        int(input(f"Staff required for {shift}: "))
    )

max_shifts = int(
    input("Maximum shifts allowed per staff: ")
)

# Availability: 1 = available, 0 = unavailable
total_slots = num_days * num_shifts
availability = np.zeros((num_staff, total_slots))

print("\nEnter availability for each staff member.")
print("Use 1 for available and 0 for unavailable.")

for i in range(num_staff):
    print(f"\nAvailability for {staff_names[i]}:")
    for j in range(total_slots):
        day = j // num_shifts + 1
        shift = shift_names[j % num_shifts]

        availability[i, j] = int(
            input(f"Day {day}, {shift}: ")
        )

# ---------- PSO PARAMETERS ----------
num_particles = int(input("Enter number of particles: "))
max_iterations = int(input("Enter maximum iterations: "))

w = 0.7
c1 = 1.5
c2 = 1.5

# Penalty weights
SHORTAGE_WEIGHT = 1000
CONFLICT_WEIGHT = 1000
OVERTIME_WEIGHT = 100
IMBALANCE_WEIGHT = 10


# ---------- CREATE A SCHEDULE ----------
def create_schedule(position):
    schedule = np.zeros(
        (num_staff, total_slots), dtype=int
    )

    for slot in range(total_slots):
        day = slot // num_shifts

        # Staff who are available and have not
        # already been assigned on this day
        candidates = [
            i for i in range(num_staff)
            if availability[i, slot] == 1
            and schedule[
                i, day * num_shifts:(day + 1) * num_shifts
            ].sum() == 0
        ]

        # Choose staff according to particle priorities
        candidates.sort(
            key=lambda i: position[i, slot]
        )

        shift_index = slot % num_shifts
        needed = required[shift_index]

        # Assign as many eligible staff as possible
        for i in candidates[:needed]:
            schedule[i, slot] = 1

    return schedule


# ---------- FITNESS FUNCTION ----------
def fitness(position):
    schedule = create_schedule(position)

    shortage = 0
    for slot in range(total_slots):
        shift_index = slot % num_shifts
        assigned = schedule[:, slot].sum()

        shortage += max(
            0, required[shift_index] - assigned
        )

    # Conflicts: more than one shift per staff per day
    conflicts = 0
    for i in range(num_staff):
        for day in range(num_days):
            start = day * num_shifts
            end = start + num_shifts

            conflicts += max(
                0, schedule[i, start:end].sum() - 1
            )

    # Overtime: shifts exceeding the allowed limit
    overtime = 0
    for i in range(num_staff):
        assigned = schedule[i, :].sum()
        overtime += max(0, assigned - max_shifts)

    # Workload imbalance: variance in total shifts
    workloads = schedule.sum(axis=1)
    imbalance = np.var(workloads)

    penalty = (
        SHORTAGE_WEIGHT * shortage
        + CONFLICT_WEIGHT * conflicts
        + OVERTIME_WEIGHT * overtime
        + IMBALANCE_WEIGHT * imbalance
    )

    return penalty, schedule


# ---------- INITIALIZE PARTICLES ----------
positions = [
    np.random.uniform(0, 1, (num_staff, total_slots))
    for _ in range(num_particles)
]

velocities = [
    np.random.uniform(-0.1, 0.1, (num_staff, total_slots))
    for _ in range(num_particles)
]

pbest_positions = [p.copy() for p in positions]
pbest_scores = [
    fitness(p)[0] for p in positions
]

best_index = np.argmin(pbest_scores)
gbest_position = pbest_positions[best_index].copy()
gbest_score = pbest_scores[best_index]


# ---------- PSO OPTIMIZATION ----------
for iteration in range(max_iterations):

    for i in range(num_particles):

        r1 = np.random.random((num_staff, total_slots))
        r2 = np.random.random((num_staff, total_slots))

        # Update velocity
        velocities[i] = (
            w * velocities[i]
            + c1 * r1 * (pbest_positions[i] - positions[i])
            + c2 * r2 * (gbest_position - positions[i])
        )

        # Update position
        positions[i] += velocities[i]

        # Evaluate the new schedule
        score, schedule = fitness(positions[i])

        # Update personal best
        if score < pbest_scores[i]:
            pbest_scores[i] = score
            pbest_positions[i] = positions[i].copy()

        # Update global best
        if score < gbest_score:
            gbest_score = score
            gbest_position = positions[i].copy()

    print(
        f"Iteration {iteration + 1}: "
        f"Best penalty = {gbest_score:.2f}"
    )


# ---------- DISPLAY BEST SCHEDULE ----------
best_penalty, best_schedule = fitness(gbest_position)

print("\n========== BEST HOSPITAL STAFF SCHEDULE ==========")

for day in range(num_days):
    print(f"\nDay {day + 1}")

    for shift_index, shift in enumerate(shift_names):
        slot = day * num_shifts + shift_index

        assigned_staff = [
            staff_names[i]
            for i in range(num_staff)
            if best_schedule[i, slot] == 1
        ]

        print(
            f"{shift}: "
            f"{', '.join(assigned_staff) if assigned_staff else 'No staff'}"
        )

print("\n========== STAFF WORKLOAD ==========")

for i in range(num_staff):
    print(
        f"{staff_names[i]}: "
        f"{int(best_schedule[i].sum())} shifts"
    )

print(f"\nMinimum penalty value: {best_penalty:.2f}")



Enter number of staff: 3
Enter name of staff 1: A
Enter name of staff 2: B
Enter name of staff 3: C
Enter number of days: 2
Enter number of shifts per day: 2
Enter name of shift 1: MORNING
Enter name of shift 2: NIGHT
Staff required for MORNING: 1
Staff required for NIGHT: 1
Maximum shifts allowed per staff: 2

Enter availability for each staff member.
Use 1 for available and 0 for unavailable.

Availability for A:
Day 1, MORNING: 1
Day 1, NIGHT: 1
Day 2, MORNING: 1
Day 2, NIGHT: 0

Availability for B:
Day 1, MORNING: 1
Day 1, NIGHT: 0
Day 2, MORNING: 1
Day 2, NIGHT: 1

Availability for C:
Day 1, MORNING: 0
Day 1, NIGHT: 1
Day 2, MORNING: 1
Day 2, NIGHT: 1
Enter number of particles: 10
Enter maximum iterations: 20
Iteration 1: Best penalty = 2.22
Iteration 2: Best penalty = 2.22
Iteration 3: Best penalty = 2.22
Iteration 4: Best penalty = 2.22
Iteration 5: Best penalty = 2.22
Iteration 6: Best penalty = 2.22
Iteration 7: Best penalty = 2.22
Iteration 8: Best penalty = 2.22
Iteration 9: